In [9]:
import darshan
import pandas as pd
from pathlib import Path

In [4]:

def get_io_performance(filename: str ,module: str="POSIX") -> pd.DataFrame:
    """
    Get the I/O performance of a module from a Darshan log file.

    Args:
        filename (str): Path to the Darshan log file.
        module (str): The darshan module to analyze (default is "POSIX").
    Returns:
        pd.DataFrame: A DataFrame containing the I/O performance metrics for each rank.
    """
    with darshan.DarshanReport(filename, read_all=True) as report:
        nprocs=report.metadata["job"]["nprocs"]

        if module in report.records:
            df=report.records[module].to_df()

            time_per_rank=df["fcounters"][[f"{module}_F_WRITE_TIME","rank"]].groupby("rank").sum()
            data_per_rank=df["counters"][[f"{module}_BYTES_WRITTEN","rank"]].groupby("rank").sum()
            io_performance = pd.merge(time_per_rank, data_per_rank, on="rank").rename(columns={f"{module}_F_WRITE_TIME":f"time_{module}",f"{module}_BYTES_WRITTEN":f"data_size_{module}"})
            io_performance[f"data_size_{module}"]/=1e+9
            io_performance[f"bandwidth_{module}"]=io_performance[f"data_size_{module}"]/io_performance[f"time_{module}"]
            return io_performance


In [ ]:
filename="/work/z19/z19/lparisi/nfs-testing/nemo/bench/experiments/lustre/EXP2026-08-05T082429/darshan_logs/lparisi_nemo.exe_id485138-485138_8-5-30444-11780130232002646330_7.darshan"

def gather_io_performance(experiments_dir: str):
    """
    Gather I/O performance metrics from Darshan log files in a specified directory.

    Args:
        experiments_dir (str): Path to the directory containing Darshan log files.
    Returns:
        pd.DataFrame: A DataFrame containing the I/O performance metrics for each rank across all log files.
    """
    # Loop over all "*.log" files in the experiments_dir
    io_performance_list = []
    for log_file in Path(experiments_dir).rglob("*.darshan"):
        io_performance=get_io_performance(str(log_file),module="H5D")["bandwidth_H5D"].sum()
        io_performance_list.append(io_performance)
    return pd.DataFrame(io_performance_list, columns=["bandwidth_H5D"] )

        


#selection=io_performance["data_size"]>=1
#io_performance[selection]["bandwidth"].sum()
io_performance

np.float64(19.183460880298476)

In [115]:
report.records["H5D"].to_df()["fcounters"]

,rank,id,H5D_F_OPEN_START_TIMESTAMP,H5D_F_READ_START_TIMESTAMP,H5D_F_WRITE_START_TIMESTAMP,H5D_F_CLOSE_START_TIMESTAMP,H5D_F_OPEN_END_TIMESTAMP,H5D_F_READ_END_TIMESTAMP,H5D_F_WRITE_END_TIMESTAMP,H5D_F_CLOSE_END_TIMESTAMP,H5D_F_READ_TIME,H5D_F_WRITE_TIME,H5D_F_META_TIME,H5D_F_MAX_READ_TIME,H5D_F_MAX_WRITE_TIME,H5D_F_FASTEST_RANK_TIME,H5D_F_SLOWEST_RANK_TIME,H5D_F_VARIANCE_RANK_TIME,H5D_F_VARIANCE_RANK_BYTES
0,3168,18125873417709877330,501.350431,0.0,0.000000,561.425676,501.350510,0.0,0.000000,561.425684,0.0,0.000000,0.000086,0.0,0.000000,0.0,0.0,0.0,0.0
1,3168,10530083211833573513,501.350574,0.0,0.000000,561.425685,501.350595,0.0,0.000000,561.425691,0.0,0.000000,0.000028,0.0,0.000000,0.0,0.0,0.0,0.0
2,3168,1643833051228441487,501.350616,0.0,0.000000,561.425692,501.350635,0.0,0.000000,561.425698,0.0,0.000000,0.000026,0.0,0.000000,0.0,0.0,0.0,0.0
3,3168,7403349132047433406,501.350677,0.0,501.356198,561.425157,501.350704,0.0,501.359809,561.425212,0.0,0.003612,0.000081,0.0,0.003612,0.0,0.0,0.0,0.0
4,3168,13006224981888598486,501.350738,0.0,501.363677,561.425221,501.350758,0.0,501.367284,561.425237,0.0,0.003607,0.000037,0.0,0.003607,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
651,3175,5035537228485846034,519.420126,0.0,519.940566,561.559560,519.420219,0.0,519.940583,561.559576,0.0,0.000017,0.000109,0.0,0.000017,0.0,0.0,0.0,0.0
652,3175,8983381615774762967,519.420245,0.0,519.940618,561.559579,519.420281,0.0,519.940627,561.559594,0.0,0.000009,0.000052,0.0,0.000009,0.0,0.0,0.0,0.0
653,3175,1196881214106646818,519.420383,0.0,519.940659,561.559599,519.420426,0.0,519.940667,561.559614,0.0,0.000008,0.000058,0.0,0.000008,0.0,0.0,0.0,0.0
654,3175,5980445058252177380,519.420447,0.0,519.765278,561.559617,519.420481,0.0,519.939754,561.559648,0.0,0.174477,0.000066,0.0,0.174477,0.0,0.0,0.0,0.0
